In [ ]:
# ==========================================
# Assignment Five - Part B1
# Loan Data Processing
# ==========================================

# Step 1: Import libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

print("Libraries imported successfully")


# ==========================================
# Step 2: Load Dataset
# ==========================================

df = pd.read_csv("raw_loan_dataset.csv")

print("\nDataset Loaded Successfully")

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())


# ==========================================
# Step 3: Clean Currency Formatting
# ==========================================

currency_cols = ["Income", "LoanAmount"]

for col in currency_cols:
    
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$","", regex=False)
        .str.replace(",","", regex=False)
    )
    
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nCurrency symbols removed")

print(df[["Income","LoanAmount"]].head())


# ==========================================
# Step 4: Fix Category Errors
# ==========================================

yes_values = ["yes","y","approved"]
no_values = ["no","n","rejected"]

for col in ["HasCollateral",
            "PreviousDefaults",
            "Approved"]:
    
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    
    df[col] = df[col].replace(
        yes_values,
        "Yes"
    )
    
    df[col] = df[col].replace(
        no_values,
        "No"
    )

    print(f"\n{col} value counts:")
    print(df[col].value_counts())


# ==========================================
# Step 5: Missing Value Imputation
# ==========================================

numeric_cols = [
    "Income",
    "CreditScore",
    "LoanAmount",
    "EmploymentYears"
]

categorical_cols = [
    "HasCollateral",
    "PreviousDefaults",
    "Approved"
]


for col in numeric_cols:
    
    median = df[col].median()
    
    df[col] = df[col].fillna(median)



for col in categorical_cols:
    
    mode = df[col].mode()[0]
    
    df[col] = df[col].fillna(mode)


print("\nMissing values after imputation:")

print(df.isnull().sum())


# ==========================================
# Step 6: Remove Duplicates
# ==========================================

before_rows = df.shape[0]

df = df.drop_duplicates()

after_rows = df.shape[0]

print("\nRows before:", before_rows)
print("Rows after:", after_rows)
print("Removed:", before_rows-after_rows)


# ==========================================
# Step 7: Outlier Handling (IQR)
# ==========================================

outlier_cols = [
    "Income",
    "CreditScore",
    "LoanAmount",
    "EmploymentYears"
]


for col in outlier_cols:
    
    Q1 = df[col].quantile(.25)
    
    Q3 = df[col].quantile(.75)

    IQR = Q3-Q1

    lower = Q1 - (1.5*IQR)
    upper = Q3 + (1.5*IQR)

    
    df[col] = df[col].clip(
        lower,
        upper
    )

print("\nOutliers capped successfully")


# ==========================================
# Step 8: Label Encoding
# ==========================================

mapping = {
    "Yes":1,
    "No":0
}


label_cols = [
    "Approved",
    "HasCollateral",
    "PreviousDefaults"
]


for col in label_cols:
    
    df[col] = df[col].map(mapping)


print("\nApproved Distribution:")
print(df["Approved"].value_counts())


# ==========================================
# Step 9: Class Balance Check
# ==========================================

print("\nClass Counts:")

print(df["Approved"].value_counts())

print("\nClass Proportions:")

print(df["Approved"].value_counts(normalize=True))


# ==========================================
# Step 10: Feature Engineering
# ==========================================

df["DebtToIncome"] = (
    df["LoanAmount"]
    /
    df["Income"]
)

df["IncomePerYearEmployed"] = (
    
    df["Income"]
    /
    (df["EmploymentYears"]+1)

)

print("\nNew Features Added")


# ==========================================
# Step 11: Feature Scaling
# ==========================================

"""
Chosen Scaler:
RobustScaler

Reason:
RobustScaler works well when
data contains outliers because
it uses median and IQR rather
than mean and standard deviation.
Loan datasets often contain
extreme income and loan values.

Source:
https://scikit-learn.org/
"""

scaler = RobustScaler()


scale_cols = [

"Income",
"CreditScore",
"LoanAmount",
"EmploymentYears",
"DebtToIncome",
"IncomePerYearEmployed"

]


df[scale_cols] = scaler.fit_transform(
    
    df[scale_cols]

)


print("\nScaling completed")


# ==========================================
# Step 12: Final Checks
# ==========================================

print("\nFinal Dataset Info:")

print(df.info())

print("\nMissing Values:")

print(df.isnull().sum())

print("\nDataset Head:")

print(df.head())


# ==========================================
# Step 13: Save Dataset
# ==========================================

df.to_csv(
    "clean_loan_dataset.csv",
    index=False
)

print(
    "\nDataset saved successfully as clean_loan_dataset.csv"
)